# My Name is Riemann Prerelease: test it, then train it

This notebook loads the 13.1 billion parameter [My Name is Riemann Prerelease](https://www.kaggle.com/models/openpeer-ai/my-name-is-riemann-prerelease) checkpoint straight from Kaggle Models, runs a handful of tests that show what the model can and cannot do yet, and then fits an NTK gate controller on data you pick. The base weights are never quantized. They are loaded in half precision (the model was trained in bf16; the T4s run fp16 natively) and sharded across the two GPUs, with gradient checkpointing so training fits in what is left over.

Before you run it:

1. Settings > Accelerator > **GPU T4 x2**. One T4 is not enough for 26 GB of weights.
2. Settings > Internet > **On** (needed for `pip install` and, optionally, to stream training data from Hugging Face).
3. The notebook looks for the model under `/kaggle/input` first (add it from the Models tab if it is not attached) and otherwise downloads it from Hugging Face, which Kaggle does at several hundred MB/s, so either way it takes a few minutes.

Loading takes about 8 to 12 minutes because the fp32 shards are read from the model mount and cast on the way in. After that each test cell runs in seconds to a minute. The whole notebook end to end is roughly 25 to 35 minutes.

Sections:

* **Load**: tokenizer, model, memory check.
* **Test**: free continuation, a token by token surprisal map, a fluency contrast table, and the stage 6 NTK controller that ships with the checkpoint.
* **Train**: fit your own NTK controller with [ntkmirror](https://github.com/leochlon/ntkmirror) (the OpenPeer AI NTK Trainer). Weights stay frozen; only a few thousand gate values are learned.
* **Optional**: a short LoRA run on the same frozen base, for comparison.

Related: the [Hugging Face model card](https://huggingface.co/OpenPeerAI/My-Name-is-Riemann-Prerelease) with the full training writeup, and three live demos ([basic](https://huggingface.co/spaces/OpenPeerAI/My-Name-is-Riemann-Prerelease-basic-demo), [OpenPeerLLM comparison](https://huggingface.co/spaces/OpenPeerAI/OpenPeerLLM-Demo), [NTK Trainer](https://huggingface.co/spaces/OpenPeerAI/NTK-Trainer-Demo)).

A word on expectations. This is a prerelease that has seen 2.3 billion tokens of pretraining data. It writes fluent text in an encyclopedic register and knows the shape of Python, but it has not been instruction tuned, so it will not chat with you. The tests below are built around what it does do.

## 1. Setup

In [ ]:
import os, sys, subprocess, time, math, gc, json, glob, shutil
NTKMIRROR = "git+https://github.com/leochlon/ntkmirror.git@da17ca3c3bbf93e97f0102d6d2d9d7355cf9727f"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", NTKMIRROR], check=True)

import torch
print("torch", torch.__version__)
n_gpu = torch.cuda.device_count()
total_free = 0
for i in range(n_gpu):
    free, total = torch.cuda.mem_get_info(i)
    total_free += free
    print(f"cuda:{i}  {torch.cuda.get_device_name(i)}  {free/2**30:.1f} GiB free of {total/2**30:.1f}")
if total_free < 25 * 2**30:
    print("\nLess than 25 GiB of GPU memory in total. The weights alone need 24.4 GiB in half precision,"
          " so part of the model will spill to CPU and everything will be very slow."
          " Switch the accelerator to GPU T4 x2 and restart.")

In [ ]:
MODEL_DIR = None
for cfg in sorted(glob.glob("/kaggle/input/**/config.json", recursive=True)):
    try:
        if json.load(open(cfg)).get("model_type") == "riemann_openpeer":
            MODEL_DIR = os.path.dirname(cfg)
            break
    except Exception:
        pass

if MODEL_DIR is None:
    free_gb = shutil.disk_usage("/tmp").free / 1e9
    print(f"Model is not attached as a Kaggle input (or the attachment failed). Downloading 52 GB from Hugging Face to /tmp ({free_gb:.0f} GB free).")
    assert free_gb > 60, "Not enough scratch space in /tmp for the download."
    os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
    from huggingface_hub import snapshot_download
    t0 = time.time()
    MODEL_DIR = snapshot_download(
        "OpenPeerAI/My-Name-is-Riemann-Prerelease",
        local_dir="/tmp/riemann",
        allow_patterns=["*.json", "*.py", "*.txt", "*.safetensors", "ntk_controller/*", "LICENSE"],
    )
    print(f"downloaded in {(time.time() - t0) / 60:.1f} min")

print("model files in", MODEL_DIR)
for f in sorted(os.listdir(MODEL_DIR)):
    p = os.path.join(MODEL_DIR, f)
    if os.path.isfile(p):
        print(f"  {f:40s} {os.path.getsize(p)/2**30:6.2f} GiB")

## 2. Load the model across both GPUs

The checkpoint ships its own `modeling_riemann.py`. One small patch is applied before loading: the class is told that a `TransformerBlock` must not be split between devices, which is what lets `accelerate` place whole layers on each T4. The memory budget leaves a couple of GiB free on each card for activations.

`DTYPE` defaults to fp16 because that is what T4 tensor cores accelerate. The sanity check two cells down verifies nothing overflowed and reloads in bf16 if it did (bf16 works on a T4 but runs several times slower). If you are on an A100 or H100, set bf16 directly.

In [ ]:
from transformers import AutoTokenizer, AutoConfig
from transformers.dynamic_module_utils import get_class_from_dynamic_module

DTYPE = torch.float16

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
tokenizer.pad_token = tokenizer.eos_token
config = AutoConfig.from_pretrained(MODEL_DIR, trust_remote_code=True)
ModelCls = get_class_from_dynamic_module(config.auto_map["AutoModelForCausalLM"], MODEL_DIR)
ModelCls._no_split_modules = ["TransformerBlock"]


def gpu_budget(reserve_first_gib=2.6, reserve_other_gib=1.8):
    budget = {}
    for i in range(torch.cuda.device_count()):
        free, _ = torch.cuda.mem_get_info(i)
        reserve = reserve_first_gib if i == 0 else reserve_other_gib
        budget[i] = max(0, int(free - reserve * 2**30))
    budget["cpu"] = 20 * 2**30
    return budget


def load_model(dtype):
    t0 = time.time()
    m = ModelCls.from_pretrained(
        MODEL_DIR, config=config, dtype=dtype, device_map="auto", max_memory=gpu_budget(),
    )
    m.eval()
    for p in m.parameters():
        p.requires_grad_(False)
    print(f"loaded in {(time.time() - t0) / 60:.1f} min")
    return m


model = load_model(DTYPE)

In [ ]:
from collections import Counter

n_params = sum(p.numel() for p in model.parameters())
placement = Counter(str(d) for d in model.hf_device_map.values())
print(f"{n_params:,} parameters in {str(DTYPE).replace('torch.', '')}")
print("modules per device:", dict(placement))
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"cuda:{i}  weights {torch.cuda.memory_allocated(i)/2**30:.1f} GiB, {free/2**30:.1f} GiB still free")
if "cpu" in placement:
    print("WARNING: some layers are on the CPU. Expect very slow steps.")

### Memory helpers and a sanity check

Two small hooks make training possible on this hardware. The first wraps every decoder block in `torch.utils.checkpoint` whenever gradients are enabled, so activations are recomputed on the backward pass instead of being stored for all 40 layers. The second makes sure the logits come back to the GPU that holds the inputs (accelerate normally does this already; the hook is a safety net). Neither changes the numbers the model produces.

Then two sentences are scored: a fluent one and the same words shuffled. A healthy load gives a low perplexity on the first and a much higher one on the second, with nothing infinite or NaN.

In [ ]:
import functools
import torch.nn.functional as F
import torch.utils.checkpoint as ckpt


def _checkpointed(orig_forward, *args, **kwargs):
    if torch.is_grad_enabled():
        return ckpt.checkpoint(orig_forward, *args, use_reentrant=False, **kwargs)
    return orig_forward(*args, **kwargs)


def _bring_logits_home(module, args, kwargs, output):
    ids = kwargs.get("input_ids", args[0] if args else None)
    if ids is not None and output.logits.device != ids.device:
        output.logits = output.logits.to(ids.device)
    return output


def prepare(m):
    for layer in m.layers:
        if not getattr(layer, "_riemann_ckpt", False):
            layer.forward = functools.partial(_checkpointed, layer.forward)
            layer._riemann_ckpt = True
    if not getattr(m, "_riemann_home_hook", False):
        m.register_forward_hook(_bring_logits_home, with_kwargs=True)
        m._riemann_home_hook = True
    return m


@torch.no_grad()
def token_surprisal(text, max_length=512):
    ids = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length).input_ids.to(model.device)
    logits = model(input_ids=ids).logits[0, :-1].float()
    targets = ids[0, 1:].to(logits.device)
    logprobs = F.log_softmax(logits, dim=-1)
    tok_lp = logprobs.gather(1, targets[:, None]).squeeze(1)
    ranks = (logprobs > tok_lp[:, None]).sum(dim=1) + 1
    nll = -tok_lp.mean().item()
    return {
        "tokens": [tokenizer.decode([t]) for t in ids[0].tolist()],
        "surprisal": [-x for x in tok_lp.tolist()],
        "rank": ranks.tolist(),
        "nll": nll,
        "ppl": math.exp(nll) if math.isfinite(nll) else float("inf"),
        "n_tokens": int(targets.numel()),
    }


model = prepare(model)

FLUENT = "The Amazon River in South America is the largest river by discharge volume of water in the world."
SALAD = "world the in water of volume discharge by river largest the is America South in River Amazon The"
r_fluent, r_salad = token_surprisal(FLUENT), token_surprisal(SALAD)
healthy = math.isfinite(r_fluent["nll"]) and math.isfinite(r_salad["nll"])
if not healthy and DTYPE == torch.float16:
    print("fp16 overflowed somewhere. Reloading in bf16; this is correct but slower on a T4.")
    del model; gc.collect(); torch.cuda.empty_cache()
    DTYPE = torch.bfloat16
    model = prepare(load_model(DTYPE))
    r_fluent, r_salad = token_surprisal(FLUENT), token_surprisal(SALAD)

print(f"fluent sentence   perplexity {r_fluent['ppl']:8.1f}")
print(f"shuffled words    perplexity {r_salad['ppl']:8.1f}")
print(f"ratio {r_salad['ppl'] / r_fluent['ppl']:.0f}x  (anything well above 10x means the load is healthy)")

## 3. Test

### 3a. Free continuation

Give it the first sentence of an article, or the start of a function, and let it keep going. Greedy decoding with a mild repetition penalty is the most readable setting for this checkpoint. The decoding loop is written out in plain PyTorch below so you can see (and change) exactly what happens. The model has no KV cache, so each new token re-reads the whole prompt; 48 tokens take 15 to 30 seconds per prompt on T4s.

In [ ]:
@torch.no_grad()
def complete(prompt, max_new_tokens=48, temperature=0.0, top_p=0.9, repetition_penalty=1.2):
    ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024 - max_new_tokens).input_ids.to(model.device)
    generated = []
    for _ in range(max_new_tokens):
        logits = model(input_ids=ids).logits[0, -1].float()
        if repetition_penalty != 1.0:
            seen = torch.unique(ids[0]).to(logits.device)
            hit = logits[seen]
            logits[seen] = torch.where(hit > 0, hit / repetition_penalty, hit * repetition_penalty)
        if temperature > 0:
            probs = torch.softmax(logits / temperature, dim=-1)
            sorted_p, sorted_i = probs.sort(descending=True)
            sorted_p[sorted_p.cumsum(0) - sorted_p > top_p] = 0.0
            next_id = sorted_i[torch.multinomial(sorted_p / sorted_p.sum(), 1)]
        else:
            next_id = logits.argmax().view(1)
        if int(next_id) == tokenizer.eos_token_id:
            break
        generated.append(int(next_id))
        ids = torch.cat([ids, next_id.view(1, 1).to(ids.device)], dim=1)
    return tokenizer.decode(generated, skip_special_tokens=True)


PROMPTS = [
    "The Nile is a major north-flowing river in northeastern Africa. It",
    "Photosynthesis is the process by which green plants",
    "def fibonacci(n):\n    \"\"\"Return the n-th Fibonacci number.\"\"\"\n",
]
for p in PROMPTS:
    t0 = time.time()
    text = complete(p)
    print("=" * 80)
    print(p + text)
    print(f"--- {time.time() - t0:.1f}s")

Try sampling by passing `temperature=0.8` to `complete`, or feed it your own opening line in the cell below.

In [ ]:
your_prompt = "The Riemann hypothesis is a conjecture in mathematics that"
print(your_prompt + complete(your_prompt, max_new_tokens=48))

### 3b. Surprisal map

Instead of asking the model to write, this asks it to read. Every token is coloured by how surprised the model was to see it given everything before it (green means expected, red means surprising). Hover a token for the exact value in nats. This is a quick way to see what the model has learned: function words and common continuations should be green, rare proper nouns red, and a factual error in otherwise fluent text often lights up.

In [ ]:
import html
from IPython.display import HTML, display


def surprisal_html(result, title=""):
    toks, surp = result["tokens"], result["surprisal"]
    cells = [f"<span style='color:#888'>{html.escape(toks[0])}</span>"]
    for t, s in zip(toks[1:], surp):
        p = min(1.0, s / 12.0)
        r, g = int(255 * p), int(200 * (1 - p))
        cells.append(f"<span title='surprisal {s:.2f} nats' style='background:rgba({r},{g},80,0.35);"
                     f"padding:1px 2px;border-radius:3px'>{html.escape(t)}</span>")
    head = (f"<div style='margin-bottom:6px'><b>{html.escape(title)}</b> perplexity <b>{result['ppl']:.1f}</b> "
            f"over {result['n_tokens']} tokens (mean surprisal {result['nll']:.2f} nats)</div>") if title else ""
    return (head + "<div style='font-family:ui-monospace,monospace;line-height:1.9;white-space:pre-wrap;"
            "word-break:break-word'>" + "".join(cells) + "</div>")


PARAGRAPH = (
    "Bernhard Riemann was a German mathematician who made profound contributions to analysis, "
    "number theory, and differential geometry. In 1859 he published a short paper on the distribution "
    "of prime numbers in which he stated what is now called the Riemann hypothesis. The paper was "
    "eight pages long and remains one of the most studied documents in mathematics."
)
display(HTML(surprisal_html(token_surprisal(PARAGRAPH), "Riemann paragraph:")))

WRONG = (
    "Bernhard Riemann was a German mathematician who made profound contributions to analysis, "
    "number theory, and differential geometry. In 1859 he published a short paper on the migration "
    "of tropical birds in which he stated what is now called the Riemann hypothesis."
)
display(HTML(surprisal_html(token_surprisal(WRONG), "Same opening, wrong second sentence:")))

### 3c. Fluency contrast

Perplexity on a handful of short texts that differ in one way at a time. A language model that has learned English should separate fluent prose from shuffled words by orders of magnitude, and should find a grammatical but false sentence only a little stranger than a true one.

In [ ]:
import pandas as pd

CONTRAST = [
    ("Fluent English, encyclopedic", "The Amazon River in South America is the largest river by discharge volume of water in the world."),
    ("Fluent English, narrative", "She closed the book, turned off the lamp, and listened to the rain against the window."),
    ("Grammatical but false", "The Amazon River is located in northern Canada and flows into the Arctic Ocean near Greenland."),
    ("Same words, shuffled", "world the in water of volume discharge by river largest the is America South in River Amazon The"),
    ("Random words", "purple elephant calculus whisper nine umbrella fjord lantern"),
    ("Python", "def add(a, b):\n    return a + b"),
    ("Arabic", "القاهرة هي عاصمة جمهورية مصر العربية وأكبر مدنها."),
]
rows = []
for label, text in CONTRAST:
    r = token_surprisal(text)
    rows.append({"text": label, "perplexity": round(r["ppl"], 1), "mean surprisal (nats)": round(r["nll"], 3), "tokens": r["n_tokens"]})
pd.DataFrame(rows).set_index("text")

### 3d. The NTK controller that ships with the checkpoint

The checkpoint includes `ntk_controller_stage6_12.85b.pt`: 6,732 signed log gates on decoder layer output channels, fitted with the NTK Trainer on 19,000 prompt and completion pairs drawn from the Fable transcripts and the jabarti finetune split. The weights it sits on are exactly the ones loaded above, so attaching it is a matter of loading a 135 KB file.

Below, 32 Fable pairs are streamed from Hugging Face (rows are long, so only the first few megabytes of the 2 GB file are read) and scored with and without the controller. About a quarter of all Fable rows were in the controller's training set, so treat this as an in-distribution check rather than a strict held-out score. The model card reports 1.323 to 1.310 nats on the real validation split.

In [ ]:
from ntkmirror import ForwardFineTuner, Example
from huggingface_hub import HfFileSystem

FABLE = "datasets/MoreThought/Fable-5.1-Max-Reasoning-Filtered-10000x/10k/train.jsonl"


def trim(text, max_tokens):
    ids = tokenizer(text, add_special_tokens=False, truncation=True, max_length=max_tokens + 1).input_ids
    return text if len(ids) <= max_tokens else tokenizer.decode(ids[:max_tokens])


def stream_fable_pairs(n, skip=0, prompt_tokens=96, completion_tokens=160):
    """Yield (first user turn, final assistant turn) pairs, trimmed so both fit in one window."""
    out = []
    fs = HfFileSystem()
    with fs.open(FABLE, "r", encoding="utf-8") as fh:
        for i, line in enumerate(fh):
            if len(out) >= n:
                break
            if i < skip or not line.strip():
                continue
            try:
                messages = json.loads(line).get("messages") or []
            except json.JSONDecodeError:
                continue
            user, assistant = None, None
            for m in messages:
                content = str(m.get("content") or "")
                if m.get("role") == "user" and user is None:
                    user = content
                elif m.get("role") == "assistant" and content.strip():
                    assistant = content
            if user and assistant:
                out.append(Example(trim(user.strip(), prompt_tokens), trim(assistant.strip(), completion_tokens)))
    return out


try:
    check_pairs = stream_fable_pairs(32, skip=0)
except Exception as e:
    print("could not stream Fable, using built-in examples instead:", e)
    from ntkmirror.data import tiny_arithmetic_eval
    check_pairs = tiny_arithmetic_eval()
print(len(check_pairs), "pairs. First one:")
print("  prompt:    ", check_pairs[0].prompt[:160].replace("\n", " "), "...")
print("  completion:", check_pairs[0].completion[:160].replace("\n", " "), "...")

In [ ]:
shipped_path = next(iter(glob.glob(os.path.join(MODEL_DIR, "ntk_controller*stage6*.pt")) +
                         glob.glob(os.path.join(MODEL_DIR, "ntk_controller", "stage6*.pt"))), None)
assert shipped_path, "stage 6 controller file not found next to the weights"

shipped = ForwardFineTuner(model, tokenizer)
shipped.load(shipped_path, allow_model_mismatch=True)
print(f"loaded {os.path.basename(shipped_path)}: {shipped.controller.raw.numel():,} gates on "
      f"{len(set(shipped.controller.layer_indices.tolist()))} of {len(model.layers)} layers")

t0 = time.time()
base = shipped.evaluate_nll(check_pairs, batch_size=4, max_length=256, use_controller=False)
ctrl = shipped.evaluate_nll(check_pairs, batch_size=4, max_length=256, use_controller=True)
print(f"scored in {time.time() - t0:.0f}s\n")
print(f"{'':28s}{'base weights':>14s}{'with controller':>18s}")
print(f"{'mean NLL (nats)':28s}{base['nll']:14.4f}{ctrl['nll']:18.4f}")
print(f"{'perplexity':28s}{math.exp(base['nll']):14.2f}{math.exp(ctrl['nll']):18.2f}")
print(f"{'next-token accuracy':28s}{base['token_acc']:14.2%}{ctrl['token_acc']:18.2%}")
print(f"\n{int(base['tokens'])} supervised tokens. Change: {ctrl['nll'] - base['nll']:+.4f} nats")

The gates are tiny multiplicative nudges (each channel is scaled by `exp(s)` with `|s| <= 0.05`), so the effect is a few hundredths of a nat, not a transformation of the model. That is the point of the method: it moves a frozen model toward a distribution without touching the weights, and the whole adjustment is a 135 KB file you can swap in and out.

## 4. Train your own NTK controller

This is the same procedure that produced the shipped controller, scaled down to a few minutes. `ForwardFineTuner` scores every layer output channel by how much the loss would move if it were gated, keeps the top `gates`, then optimises those log gates with AdamW while the 13.1 billion weights stay frozen.

Pick your data with `TRAIN_SOURCE`:

* `"fable"` streams `N_TRAIN + N_VAL` fresh pairs from the Fable transcripts (different rows from the check above). These are agentic coding sessions: the prompt is the head of a long task briefing and the completion is the start of the agent's final report, so expect dense technical text rather than trivia.
* `"custom"` uses `CUSTOM_PAIRS` below. Replace them with your own prompt and completion pairs; a few dozen is enough to see movement. Completions should start with a space or newline so the tokenizer treats them as a continuation.

With `batch_size=4` and `max_length=256`, one step is a forward and a checkpointed backward through the whole model and takes 3 to 5 seconds on fp16 T4s. Forty steps plus gate selection and two validation passes is about four minutes.

In [ ]:
TRAIN_SOURCE = "fable"
N_TRAIN, N_VAL = 64, 16

CUSTOM_PAIRS = [
    ("Q: What is the capital of France?\nA:", " Paris."),
    ("Q: Which planet is closest to the Sun?\nA:", " Mercury."),
    ("Q: What gas do plants absorb from the air?\nA:", " Carbon dioxide."),
    ("Q: Who wrote Pride and Prejudice?\nA:", " Jane Austen."),
    ("Q: What is the boiling point of water at sea level in Celsius?\nA:", " 100 degrees."),
    ("Q: What is the largest ocean on Earth?\nA:", " The Pacific Ocean."),
    ("Q: How many legs does a spider have?\nA:", " Eight."),
    ("Q: What is the chemical symbol for gold?\nA:", " Au."),
    ("Q: Which continent is Egypt in?\nA:", " Africa."),
    ("Q: What is the square root of 81?\nA:", " Nine."),
    ("Q: Who painted the Mona Lisa?\nA:", " Leonardo da Vinci."),
    ("Q: What is the hardest natural substance?\nA:", " Diamond."),
    ("Q: In which year did the Second World War end?\nA:", " 1945."),
    ("Q: What is the smallest prime number?\nA:", " Two."),
    ("Q: What language is spoken in Brazil?\nA:", " Portuguese."),
    ("Q: Which organ pumps blood through the body?\nA:", " The heart."),
    ("Q: What is the freezing point of water in Fahrenheit?\nA:", " 32 degrees."),
    ("Q: What is the longest river in Africa?\nA:", " The Nile."),
    ("Q: How many sides does a hexagon have?\nA:", " Six."),
    ("Q: What is the currency of Japan?\nA:", " The yen."),
]

if TRAIN_SOURCE == "fable":
    pairs = stream_fable_pairs(N_TRAIN + N_VAL, skip=40)
else:
    pairs = [Example(p, c) for p, c in CUSTOM_PAIRS]
    N_VAL = max(4, len(pairs) // 5)
train_pairs, val_pairs = pairs[:-N_VAL], pairs[-N_VAL:]
print(f"{len(train_pairs)} training pairs, {len(val_pairs)} validation pairs from {TRAIN_SOURCE}")

# Unrelated prose, used to check that the controller does not degrade everything else.
NEUTRAL = [
    "The city council approved the budget after a three hour meeting on Tuesday evening.",
    "Glaciers form where snow accumulates faster than it melts over many years.",
    "The recipe calls for two cups of flour, one egg, and a pinch of salt.",
    "Copper conducts electricity well, which is why it is used in household wiring.",
    "The museum reopened in April with a new wing devoted to photography.",
    "A leap year has 366 days, with the extra day added at the end of February.",
    "The train from Lyon arrives at platform four a little after nine.",
    "Honeybees communicate the location of flowers with a dance inside the hive.",
]
neutral_examples = [Example("", t) for t in NEUTRAL]

In [ ]:
GATES = 1024          # how many layer output channels get a gate (the shipped controller has 6,732)
STEPS = 40
LR = 5e-3
MAX_LOG_GATE = 0.05   # each gate scales its channel by exp(s) with |s| <= this

if "shipped" in globals() and shipped.controller is not None:
    shipped.controller.remove()
gc.collect(); torch.cuda.empty_cache()

tuner = ForwardFineTuner(model, tokenizer, gates=GATES, max_log_gate=MAX_LOG_GATE)

before_val = tuner.evaluate_nll(val_pairs, batch_size=4, max_length=256, use_controller=False)
before_neutral = tuner.evaluate_nll(neutral_examples, batch_size=4, max_length=256, use_controller=False)

t0 = time.time()
report = tuner.fit(
    train_pairs, steps=STEPS, lr=LR, batch_size=4, max_length=256, score_batches=8,
    validation_examples=val_pairs, validation_interval=10, verbose=True,
)
print(f"\nfit took {(time.time() - t0) / 60:.1f} min, {int(report['selected_gates'])} gates, "
      f"training loss {report['loss_first']:.4f} -> {report['loss_last']:.4f}")

after_val = tuner.evaluate_nll(val_pairs, batch_size=4, max_length=256, use_controller=True)
after_neutral = tuner.evaluate_nll(neutral_examples, batch_size=4, max_length=256, use_controller=True)

print(f"\n{'':26s}{'base':>10s}{'controller':>12s}{'change':>10s}")
for name, b, a in (("validation pairs", before_val, after_val), ("unrelated prose", before_neutral, after_neutral)):
    print(f"{name:26s}{b['nll']:10.4f}{a['nll']:12.4f}{a['nll'] - b['nll']:+10.4f}  nats")
print(f"\nvalidation next-token accuracy {before_val['token_acc']:.2%} -> {after_val['token_acc']:.2%}")
print(f"trainable values: {int(report['selected_gates']):,} of {n_params:,} parameters "
      f"({report['selected_gates'] / n_params:.2e} of the model)")

A good run lowers NLL on the validation pairs more than on unrelated prose. If both move by the same amount the controller has learned something generic (often just a bias toward the completion format); if validation gets worse while training loss falls, the run is overfitting and you want more pairs or fewer steps.

The plot shows what was learned: the spread of gate values and how many gates landed on each layer.

In [ ]:
import matplotlib.pyplot as plt

raw = tuner.controller.s.detach().float().cpu().numpy()
layers = tuner.controller.layer_indices.detach().cpu().tolist()
per_layer = Counter(layers)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].hist(raw, bins=40, color="#4c6ef5")
axes[0].set_title("learned log-gate values s  (channel x exp(s))")
axes[0].set_xlabel("s")
axes[1].bar(sorted(per_layer), [per_layer[k] for k in sorted(per_layer)], color="#20c997")
axes[1].set_title("gates per decoder layer")
axes[1].set_xlabel("layer")
fig.tight_layout()
plt.show()

### Generate with and without your controller

Same prompt, same decoding settings, base weights first and then with the gates attached.

In [ ]:
probe = val_pairs[0].prompt if TRAIN_SOURCE == "fable" else "Q: What is the tallest mountain on Earth?\nA:"
print("PROMPT:", probe[:300].replace("\n", " "), "\n")
print("BASE:      ", complete(probe, max_new_tokens=40).strip()[:400], "\n")
with tuner.controller.attached():
    print("CONTROLLER:", complete(probe, max_new_tokens=40).strip()[:400])

## 5. Save and reuse

The controller is saved to `/kaggle/working`, so it appears under Output when you save a version of this notebook. To use it later, load the model the same way and call `load` on a fresh `ForwardFineTuner`; the file records which layers and channels it gates, so it only works on this checkpoint.

In [ ]:
OUT_DIR = "/kaggle/working"
ctrl_path = os.path.join(OUT_DIR, "riemann_ntk_controller.pt")
tuner.save(ctrl_path, metadata={
    "source": "kaggle test_and_train notebook", "train_source": TRAIN_SOURCE,
    "train_pairs": len(train_pairs), "steps": STEPS, "lr": LR, "gates": int(report["selected_gates"]),
    "validation_nll_base": before_val["nll"], "validation_nll_controller": after_val["nll"],
})
with open(os.path.join(OUT_DIR, "riemann_ntk_controller.json"), "w") as fh:
    json.dump({k: v for k, v in report.items() if not isinstance(v, list)}, fh, indent=2)
print(f"saved {ctrl_path} ({os.path.getsize(ctrl_path) / 1024:.0f} KB)")

# Round trip: a fresh tuner, load from disk, same numbers.
reloaded = ForwardFineTuner(model, tokenizer).load(ctrl_path, allow_model_mismatch=True)
check = reloaded.evaluate_nll(val_pairs, batch_size=4, max_length=256, use_controller=True)
print(f"reloaded controller validation NLL {check['nll']:.4f} (fitted: {after_val['nll']:.4f})")
reloaded.controller.remove()

## 6. Optional: LoRA on the same frozen base

For comparison, here is a short LoRA run with [peft](https://github.com/huggingface/peft) on the same training pairs. Rank 8 adapters on the attention query and value projections add about 6.5 million trainable parameters, roughly 6,000 times more than the controller above, and the adapter on disk is about 25 MB rather than 135 KB. Set `RUN_LORA = False` to skip it.

The base weights stay frozen and in half precision; the adapters are kept in fp32 and a loss scaler guards the fp16 backward pass.

In [ ]:
RUN_LORA = True
LORA_STEPS, LORA_LR = 30, 2e-4

if RUN_LORA:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "peft"], check=True)
    from peft import LoraConfig, get_peft_model
    from ntkmirror.data import make_batch, batches
    from ntkmirror.losses import causal_loss_from_logits

    if tuner.controller is not None:
        tuner.controller.remove()
    gc.collect(); torch.cuda.empty_cache()

    lora_cfg = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.0, bias="none",
                          target_modules=["query", "value"], task_type="CAUSAL_LM")
    lora = get_peft_model(model, lora_cfg)
    lora.print_trainable_parameters()

    params = [p for p in lora.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=LORA_LR, weight_decay=0.0)
    scaler = torch.amp.GradScaler("cuda", enabled=(DTYPE == torch.float16))
    train_batches = list(batches(train_pairs, 4))

    def lora_eval(examples):
        with torch.no_grad():
            tot, n = 0.0, 0
            for chunk in batches(examples, 4):
                b = make_batch(tokenizer, chunk, device=model.device, max_length=256)
                out = lora(input_ids=b["input_ids"], attention_mask=b["attention_mask"])
                loss = causal_loss_from_logits(out.logits, b["labels"])
                k = int((b["labels"][:, 1:] != -100).sum())
                tot += loss.item() * k; n += k
            return tot / max(1, n)

    with lora.disable_adapter():
        lora_before = lora_eval(val_pairs)

    t0 = time.time()
    lora.eval()  # keeps dropout off; gradients still flow to the adapters
    for step in range(LORA_STEPS):
        b = make_batch(tokenizer, train_batches[step % len(train_batches)], device=model.device, max_length=256)
        with torch.enable_grad():
            out = lora(input_ids=b["input_ids"], attention_mask=b["attention_mask"])
            loss = causal_loss_from_logits(out.logits, b["labels"])
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update()
        if step == 0 or (step + 1) % 10 == 0:
            print(f"step {step + 1:3d}/{LORA_STEPS}  loss {loss.item():.4f}  ({time.time() - t0:.0f}s)")

    lora_after = lora_eval(val_pairs)
    print(f"\nvalidation NLL  base {lora_before:.4f}  ->  LoRA {lora_after:.4f}  ({lora_after - lora_before:+.4f} nats)")
    print(f"NTK controller reached {after_val['nll']:.4f} with {int(report['selected_gates']):,} trainable values")

    lora.save_pretrained(os.path.join(OUT_DIR, "riemann_lora_r8"))
    print("adapter saved to", os.path.join(OUT_DIR, "riemann_lora_r8"))
    model = lora.unload()   # strip the adapter layers so the base model is clean again
    del lora, opt; gc.collect(); torch.cuda.empty_cache()

## What to try next

* Point `TRAIN_SOURCE = "custom"` at pairs from your own domain and raise `STEPS` to 100 or more. The shipped controller used 240 steps on 19,000 pairs.
* `GATES` trades capacity for selectivity. Fewer gates move less but also leak less onto unrelated text.
* `ForwardFineTuner.fit` also accepts `retain_examples` with `retain_weight` or `kl_to_base` to explicitly anchor the model on text you do not want to change.
* Pretraining the base further is out of scope for a T4 session; the full pipeline, including the growth ladder and the 8x MI325X stage scripts, is in the `training/` folder of the Hugging Face repository.